# Lab 6 - Task 3: Unsharp Masking (cv2 approach)

**Theory:**  
Unsharp masking enhances edges by subtracting a blurred version of the image from the original, making the image appear sharper.

**Steps:**
1. Apply Gaussian blur to the image.
2. Subtract the blurred image from the original to highlight edges.
3. Combine the original and edge-highlighted image using `cv2.addWeighted()`.
4. Adjust the weighting to see different sharpness levels.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Load Image

In [ ]:
image = cv2.imread('image.png', cv2.IMREAD_GRAYSCALE)

if image is None:
    image = np.random.randint(50, 200, (256, 256), dtype=np.uint8)
    print("No image file found — using a synthetic test image.")

## Part 3A: Basic Unsharp Masking

In [ ]:
# Step 1: Blur the image
blurred = cv2.GaussianBlur(image, (9, 9), 10)

# Step 2 & 3: Subtract blurred from original and combine
# Formula: sharpened = alpha * original + beta * blurred + gamma
# With alpha=1.5, beta=-0.5: sharpened = original + 0.5*(original - blurred)
unsharp_image = cv2.addWeighted(image, 1.5, blurred, -0.5, 0)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(blurred, cmap='gray')
axes[1].set_title('Blurred (Gaussian 9x9, σ=10)')
axes[1].axis('off')
axes[2].imshow(unsharp_image, cmap='gray')
axes[2].set_title('Unsharp Masked (α=1.5, β=-0.5)')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## Part 3B: Visualizing the Edge Mask

In [ ]:
# Edge mask = original - blurred  (high-frequency detail)
edge_mask = cv2.subtract(image, blurred)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(edge_mask, cmap='gray')
axes[1].set_title('Edge Mask (Original − Blurred)')
axes[1].axis('off')
axes[2].imshow(unsharp_image, cmap='gray')
axes[2].set_title('Sharpened')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## Part 3C: Different Alpha/Beta Weights

In [ ]:
# alpha + beta should equal 1 to preserve average brightness
# Higher alpha (> 1) with negative beta sharpens more aggressively
weights = [
    (1.0, 0.0,  'Original (no sharpening)'),
    (1.2, -0.2, 'Mild sharpening α=1.2'),
    (1.5, -0.5, 'Medium sharpening α=1.5'),
    (2.0, -1.0, 'Strong sharpening α=2.0'),
]

blurred_base = cv2.GaussianBlur(image, (9, 9), 10)

fig, axes = plt.subplots(1, len(weights), figsize=(18, 5))
for ax, (alpha, beta, title) in zip(axes, weights):
    result = cv2.addWeighted(image, alpha, blurred_base, beta, 0)
    ax.imshow(result, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Unsharp Masking — Varying Weights (Gaussian 9x9, σ=10)', fontsize=11)
plt.tight_layout()
plt.show()

## Part 3D: Different Blur Kernel Sizes

In [ ]:
kernel_sizes = [3, 5, 9, 15]

fig, axes = plt.subplots(1, len(kernel_sizes) + 1, figsize=(20, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for ax, k in zip(axes[1:], kernel_sizes):
    blurred_k = cv2.GaussianBlur(image, (k, k), 0)
    sharpened = cv2.addWeighted(image, 1.5, blurred_k, -0.5, 0)
    ax.imshow(sharpened, cmap='gray')
    ax.set_title(f'Unsharp Mask\nKernel {k}x{k}')
    ax.axis('off')

plt.suptitle('Unsharp Masking — Varying Blur Kernel Size (α=1.5, β=-0.5)', fontsize=11)
plt.tight_layout()
plt.show()

## Discussion

**Unsharp masking formula:**
$$\text{sharpened} = \alpha \cdot \text{original} + \beta \cdot \text{blurred}, \quad \alpha - \beta = 1$$

Equivalently: $\text{sharpened} = \text{original} + (\alpha - 1)(\text{original} - \text{blurred})$

- **Larger blur kernel** → captures lower-frequency content → sharpening amplifies broader edge structures.
- **Higher alpha** → stronger sharpening but may introduce noise amplification and halo artifacts.
- **Smaller blur kernel** → fine detail is amplified, micro-texture enhanced.